# Loading in and quality controlling the enrollment data

In [0]:
#Reading in the enrollment data to identify cut-off dates (based on who was under 18 when they signed up). 
!pip install openpyxl
import pandas as pd

enroll_info_path = "/Workspace/Users/user/user/UMD Enrollment Log_11.7.2025_for_Gabriel.xlsx"

enroll_info = pd.read_excel(enroll_info_path)
enroll_info.head(5)

In [0]:
# Rows with NaT in either date column
na_rows = enroll_info[enroll_info['Date of Enrollment'].isnull() | enroll_info['D.O.B.'].isnull()]
print(f"Rows with missing dates: {len(na_rows)}")
display(na_rows)

In [0]:
# Filter them out
enroll_info_filtered = enroll_info.dropna(subset=['Date of Enrollment', 'D.O.B.']).copy()
print(f"Remaining rows: {len(enroll_info_filtered)}")

In [0]:
# Count unique types per column
type_counts = {}
for col in enroll_info_filtered.columns:
    type_counts[col] = enroll_info_filtered[col].apply(type).nunique()

# Show columns with mixed types
mixed_cols = {k: v for k, v in type_counts.items() if v > 1}
print(f"Columns with mixed types: {len(mixed_cols)}")
print(mixed_cols)

# Display type breakdown for each mixed column
for col in mixed_cols:
    print(f"\n--- {col} ---")
    print(enroll_info_filtered[col].apply(type).value_counts())

In [0]:
#Seeing the rows where there are non-float values in the Age at Enrollment column
enroll_info_filtered_AoE_non_float_rows = enroll_info_filtered[enroll_info_filtered['Age at Enrollment'].apply(lambda x: not isinstance(x, float))].copy()
enroll_info_filtered_AoE_non_float_rows['Age at Enrollment'] = enroll_info_filtered_AoE_non_float_rows['Age at Enrollment'].astype(str)
display(enroll_info_filtered_AoE_non_float_rows)

In [0]:
#All values above seem reasonable to keep, especially since we are manually calculating age at enrollment later anyway
enroll_info_filtered_AoE_float = enroll_info_filtered.copy()
enroll_info_filtered_AoE_float['Age at Enrollment'] = pd.to_numeric(enroll_info_filtered_AoE_float['Age at Enrollment'], errors='coerce')
display(enroll_info_filtered_AoE_float)

In [0]:
# Double checking previously odd values to ensure they are now floats
enroll_info_filtered_AoE_previously_non_float_rows = enroll_info_filtered_AoE_float[enroll_info_filtered_AoE_float['Subject ID'].isin(enroll_info_filtered_AoE_non_float_rows['Subject ID'])]
display(enroll_info_filtered_AoE_previously_non_float_rows)

In [0]:
#checking 5 different values for both floats and strings for column Enrollment Status as a quality check
for dtype in enroll_info_filtered['Enrollment Status'].apply(type).unique():
    enrollment_status_vals = enroll_info_filtered[enroll_info_filtered['Enrollment Status'].apply(lambda x: isinstance(x, dtype))]['Enrollment Status'].head(5).tolist()
    print(f"\n--- {dtype} ---")
    print(enrollment_status_vals)

In [0]:
#Checking each unique value for the floats in that column to quality check that they are all na values
enrollment_status_float_vals = enroll_info_filtered[enroll_info_filtered['Enrollment Status'].apply(lambda x: isinstance(x, float))]['Enrollment Status'].unique()
print(enrollment_status_float_vals)

In [0]:
#Setting enroll_info_filtered_AoE_float to be just enroll_info_filtered2 for easier reference since there's no need to change the Enrollment Status values
enroll_info_filtered2 = enroll_info_filtered_AoE_float
display(enroll_info_filtered2)

In [0]:
# Look for duplicate MRNs 
duplicate_mrns = enroll_info_filtered2[enroll_info_filtered2['Epic MRN'].duplicated(keep=False)].sort_values('Epic MRN')
print(f"Rows with duplicate MRNs: {len(duplicate_mrns)}")
display(duplicate_mrns)

In [0]:
# For both pairs of duplicate MRNs, the birthday is the same for both individuals. This makes it likely that they are the same individual, just enrolled twice for some reason. Removing latter enrollment event for each. 
# Confirmed that these are duplicate entries. 

enroll_info_filtered3 = enroll_info_filtered2.sort_values('Date of Enrollment').drop_duplicates(subset='Epic MRN', keep='first').copy()
print(f"Before: {len(enroll_info_filtered2)} rows")
print(f"After: {len(enroll_info_filtered3)} rows")
# Rows in filtered2 but not in filtered3
dropped_rows_enroll_info_filtered23 = enroll_info_filtered2[~enroll_info_filtered2.index.isin(enroll_info_filtered3.index)]
display(dropped_rows_enroll_info_filtered23)

In [0]:
# Splitting those enrolled as adults and those enrolled as minors
import pandas as pd
from dateutil.relativedelta import relativedelta

# Parse as datetime
enroll_info_filtered3['Date of Enrollment'] = pd.to_datetime(enroll_info_filtered3['Date of Enrollment'], format='%Y-%m-%d')
enroll_info_filtered3['D.O.B.'] = pd.to_datetime(enroll_info_filtered3['D.O.B.'], format='%Y-%m-%d')

# Calculate exact age at enrollment using relativedelta
enroll_info_filtered3['age_at_enrollment'] = enroll_info_filtered3.apply(
    lambda row: relativedelta(row['Date of Enrollment'], row['D.O.B.']).years, axis=1
)

# Split into two dataframes (those joined after 18 and those joined before 18)
enrolled_as_adults = enroll_info_filtered3[enroll_info_filtered3['age_at_enrollment'] >= 18].copy()
enrolled_as_minors = enroll_info_filtered3[enroll_info_filtered3['age_at_enrollment'] < 18].copy()

# Add 18th birthday column for those who joined as minors
enrolled_as_minors['18th_birthday'] = enrolled_as_minors['D.O.B.'] + pd.DateOffset(years=18)

print(f"Enrolled at 18+: {len(enrolled_as_adults)} rows")
print(f"Enrolled under 18: {len(enrolled_as_minors)} rows")

display(enrolled_as_adults.head())
display(enrolled_as_minors.head())

In [0]:
# Combining enrolled_as_minors and enrolled_as_adults into enrolled_full
enrolled_full = pd.concat([enrolled_as_minors, enrolled_as_adults], ignore_index=True)
print(f"Minors: {len(enrolled_as_minors)} + Adults: {len(enrolled_as_adults)} = Total: {len(enrolled_full)}")

# Loading in and filtering notes

In [0]:
#Reading in the notes
import pandas as pd

notes_path = "/Workspace/Users/user/user/umdhpo_notes_202605141230.csv"

notes = pd.read_csv(notes_path)
notes.head(5)

In [0]:
# Filtering out those enrolled before 18 whose note was written after their 18th birthday and joining back together the qualifying portion of the cohort
import pandas as pd
from dateutil.relativedelta import relativedelta

# Ensure datetime
notes['NOTE_FILE_TIME_DTTM'] = pd.to_datetime(notes['NOTE_FILE_TIME_DTTM'])

# Left merge to keep all rows including adults
notes_merged = notes.merge(
    enrolled_as_minors[['Epic MRN', '18th_birthday']],
    left_on='PAT_MRN_ID',
    right_on='Epic MRN',
    how='left'
)

# Display the rows being removed (minors whose notes were filed after they turned 18)
filed_after_18 = notes_merged[
    notes_merged['18th_birthday'].notnull() & 
    (notes_merged['18th_birthday'] < notes_merged['NOTE_FILE_TIME_DTTM'])
]
display(filed_after_18)

# Keep everything else (adults + minors whose notes were filed before they turned 18)
notes_filtered1 = notes_merged[
    notes_merged['18th_birthday'].isnull() | 
    (notes_merged['NOTE_FILE_TIME_DTTM'] <= notes_merged['18th_birthday'])
].drop(columns=['Epic MRN', '18th_birthday']).copy()
display(notes_filtered1)

# Sanity check the math
print(f"Original notes: {len(notes)}")
print(f"Filtered notes: {len(notes_filtered1)}")
print(f"Notes filed after 18th birthday: {len(filed_after_18)}")
print(f"Confirm Original Notes - Filtered Notes - Notes filed after 18th birthday = 0:{len(notes) - len(notes_filtered1) - len(filed_after_18)}")

In [0]:
#Identifying unique values for author role to filter out irrelevant role notes. 
notes_filtered1['AUTHOR_ROLE'].value_counts()

In [0]:
#dropping rows where clinician type is not 'Physician', 'Resident', 'Nurse Practitioner', 'Fellow', or 'Medical Student' and then displaying the dropped rows
valid_roles = ['Physician', 'Resident', 'Nurse Practitioner', 'Fellow', 'Medical Student']
notes_filtered2 = notes_filtered1[notes_filtered1['AUTHOR_ROLE'].isin(valid_roles)].copy()

print(f"Before: {len(notes_filtered1)} rows")
print(f"After: {len(notes_filtered2)} rows")
dropped_roles_notes_filtered_12 = notes_filtered1[~notes_filtered1['AUTHOR_ROLE'].isin(valid_roles)]
display(dropped_roles_notes_filtered_12)

# Joining enrollment data, notes, and date of last review data then further filtering

In [0]:
# Dropping any rows from notes where there is not a matching MRN in the enrollment dataframe
# Combine all valid MRNs from both groups
valid_mrns = pd.concat([
    enrolled_full['Epic MRN']
]).unique()

# Rows being dropped (no matching MRN)
dropped_notes_matching = notes_filtered2[~notes_filtered2['PAT_MRN_ID'].isin(valid_mrns)].copy()
print(f"Rows dropped: {len(dropped_notes_matching)}")
dropped_notes_matching.head()

# Keep only notes with matching MRNs
notes_filtered3 = notes_filtered2[notes_filtered2['PAT_MRN_ID'].isin(valid_mrns)].copy()

print(f"notes_filtered2: {len(notes_filtered2)} rows")
print(f"notes_filtered3: {len(notes_filtered3)} rows")
display(notes_filtered3)

In [0]:
#Combining notes and enrollment dataframes 
notes_enrolled = notes_filtered3.merge(
    enrolled_full,
    left_on='PAT_MRN_ID',
    right_on='Epic MRN',
    how='inner'
)
print(f"Joined rows: {len(notes_enrolled)}")
display(notes_enrolled.head())

In [0]:
# Loading in redcap data
redcap_data = pd.read_csv('/Workspace/Users/user/user/UndiagnosedMendelian-EHRExtraction_DATA_LABELS_2026-05-26_1456.csv')
display(redcap_data.head())
print(len(redcap_data))

In [0]:
enrollment_log = pd.read_excel(
    "/Workspace/Users/user/user/UMD Enrollment Log_11.7.2025_for_Gabriel.xlsx",
    dtype={'Age at Enrollment': str}
)
display(enrollment_log)

In [0]:
# Merge with indicator to track matches
notes_enrolled2 = notes_enrolled.merge(
    redcap_data[['Record ID #:', 'Date of last chart review:']],
    left_on='Subject ID',
    right_on='Record ID #:',
    how='outer',
    indicator=True
)

# Fill in PAT_MRN_ID for rows that only have a Subject ID/Record ID
# by looking up the Epic MRN in enrollment_log
sid_to_mrn = enrollment_log[['Subject ID', 'Epic MRN']].dropna().copy()
sid_to_mrn['Subject ID'] = sid_to_mrn['Subject ID'].astype(str).str.strip()
sid_to_mrn['Epic MRN'] = sid_to_mrn['Epic MRN'].astype(str).str.strip()
sid_to_mrn = sid_to_mrn.drop_duplicates(subset='Subject ID')

# Use Subject ID or Record ID to look up MRN where PAT_MRN_ID is missing
lookup_id = notes_enrolled2['Subject ID'].fillna(notes_enrolled2['Record ID #:'])
lookup_id_clean = lookup_id.str.strip()
mrn_fill = lookup_id_clean.map(sid_to_mrn.set_index('Subject ID')['Epic MRN'])

notes_enrolled2['PAT_MRN_ID'] = notes_enrolled2['PAT_MRN_ID'].fillna(mrn_fill)

# Flag unmatched minors and adults
minor_ids = enrolled_as_minors['Subject ID'].unique()
adult_ids = enrolled_as_adults['Subject ID'].unique()

# MRNs that successfully matched both sides
both_mrns = set(notes_enrolled2[notes_enrolled2['_merge'] == 'both']['PAT_MRN_ID'].dropna())

# Unmatched minors in notes_enrolled
no_match_notes_minors = notes_enrolled2[
    (notes_enrolled2['_merge'] == 'left_only') & 
    (notes_enrolled2['Subject ID'].isin(minor_ids))
]
print(f"Minor notes with no REDCap match: {len(no_match_notes_minors)}")
display(no_match_notes_minors)

# Unmatched adults in notes_enrolled
no_match_notes_adults = notes_enrolled2[
    (notes_enrolled2['_merge'] == 'left_only') & 
    (notes_enrolled2['Subject ID'].isin(adult_ids))
]
print(f"Adult notes with no REDCap match: {len(no_match_notes_adults)}")
display(no_match_notes_adults)

# Unmatched REDCap rows
no_match_redcap = notes_enrolled2[
    (notes_enrolled2['_merge'] == 'right_only') & 
    (notes_enrolled2['Record ID #:'])
]
print(f"REDCap participants with no notes match: {len(no_match_redcap)}")
display(no_match_redcap)

# --- Reporting: truly lost MRNs (not in any 'both' matched row) ---
minor_mrns = set(no_match_notes_minors['PAT_MRN_ID'].dropna())
adult_mrns = set(no_match_notes_adults['PAT_MRN_ID'].dropna())
redcap_mrns = set(no_match_redcap['PAT_MRN_ID'].dropna())

print(f"\nTotal rows (nothing dropped): {len(notes_enrolled2)}")
print(f"Unique MRNs: {notes_enrolled2['PAT_MRN_ID'].nunique()}")
print(f"\nMinor notes with no REDCap match — unique MRNs: {len(minor_mrns)} (truly lost: {len(minor_mrns - both_mrns)})")
print(f"Adult notes with no REDCap match — unique MRNs: {len(adult_mrns)} (truly lost: {len(adult_mrns - both_mrns)})")
print(f"REDCap with no notes match — unique MRNs: {len(redcap_mrns)} (truly lost: {len(redcap_mrns - both_mrns)})")
print(f"\nTotal truly lost unique MRNs: {len((minor_mrns | adult_mrns | redcap_mrns) - both_mrns)}")

In [0]:
# Normalize helper for stripping spaces
def strip_clean(series):
    return series.astype(str).str.strip()

# Step 1: Build MRN -> Date of last chart review lookup from rows that have both
mrn_to_review = notes_enrolled2[notes_enrolled2['Date of last chart review:'].notna()].copy()
mrn_to_review['mrn_clean'] = strip_clean(mrn_to_review['PAT_MRN_ID'])
mrn_to_review['Date of last chart review:'] = pd.to_datetime(mrn_to_review['Date of last chart review:'])
mrn_to_review = (
    mrn_to_review
    .sort_values('Date of last chart review:', ascending=False)
    .drop_duplicates(subset='mrn_clean', keep='first')
    [['mrn_clean', 'Date of last chart review:']]
)

# Step 2: Identify unmatched notes (have MRN but missing Date of last chart review)
notes_enrolled2['mrn_clean'] = strip_clean(notes_enrolled2['PAT_MRN_ID'])
unmatched_mask = (
    notes_enrolled2['PAT_MRN_ID'].notna() & 
    notes_enrolled2['Date of last chart review:'].isna()
)

# Step 3: Fill in Date of last chart review for unmatched notes using MRN lookup
fill_values = notes_enrolled2.loc[unmatched_mask, 'mrn_clean'].map(
    mrn_to_review.set_index('mrn_clean')['Date of last chart review:']
)
notes_enrolled2.loc[unmatched_mask, 'Date of last chart review:'] = fill_values

# --- Call out rows that matched on both but still have NA in key columns ---
check_cols = ['PAT_MRN_ID', 'Date of last chart review:', 'NOTE_FILE_TIME_DTTM', 'NOTE_TEXT']

matched_both = notes_enrolled2[notes_enrolled2['_merge'] == 'both']

# MRNs that survive (have all four columns populated in at least one row)
clean_mrns = set(
    notes_enrolled2.dropna(subset=check_cols)['PAT_MRN_ID'].dropna().unique()
)

print("Unique MRNs truly lost due to NA despite matching on both:")
for col in check_cols:
    na_in_col = matched_both[matched_both[col].isna()]
    truly_lost = set(na_in_col['PAT_MRN_ID'].dropna().unique()) - clean_mrns
    print(f"  {col}: {len(truly_lost)} unique MRNs")

any_na_mask = matched_both[check_cols].isna().any(axis=1)
all_na_mrns = set(matched_both[any_na_mask]['PAT_MRN_ID'].dropna().unique())
truly_lost_total = all_na_mrns - clean_mrns
print(f"  Total truly lost: {len(truly_lost_total)} unique MRNs")

# Step 4: Drop rows that are still unmatched after the fill
notes_enrolled2_matched = notes_enrolled2.dropna(
    subset=check_cols
).copy()

notes_enrolled3 = notes_enrolled2_matched.drop(columns=['mrn_clean', '_merge']).reset_index(drop=True)

# --- Reporting ---
filled_count = fill_values.notna().sum()
filled_mrns = notes_enrolled2.loc[unmatched_mask & fill_values.notna(), 'PAT_MRN_ID'].nunique()
still_unmatched = unmatched_mask.sum() - filled_count
dropped = len(notes_enrolled2) - len(notes_enrolled3)

print(f"\nUnmatched notes filled via MRN lookup: {filled_count} rows ({filled_mrns} unique MRNs saved)")
print(f"Still unmatched after MRN lookup: {still_unmatched} rows")
print(f"Total rows dropped (no match possible): {dropped}")
print(f"\nnotes_enrolled2 rows: {len(notes_enrolled2)}")
print(f"notes_enrolled3 rows: {len(notes_enrolled3)}")
print(f"notes_enrolled2 unique MRNs: {notes_enrolled2['PAT_MRN_ID'].nunique()}")
print(f"notes_enrolled3 unique MRNs: {notes_enrolled3['PAT_MRN_ID'].nunique()}")
print(f"Unique MRNs lost: {notes_enrolled2['PAT_MRN_ID'].nunique() - notes_enrolled3['PAT_MRN_ID'].nunique()}")

In [0]:
import pandas as pd
both_missing_review = matched_both[matched_both['Date of last chart review:'].isna()]
print(f"\nMatched on both but missing Date of last chart review: {len(both_missing_review)} rows ({both_missing_review['PAT_MRN_ID'].nunique()} unique MRNs)")
pd.set_option('display.max_rows', None)
display(both_missing_review)

In [0]:
unique_ids = sorted(both_missing_review['Record ID #:'].unique())
print(f"Unique Record IDs: {len(unique_ids)}")
display(unique_ids)

## Troubleshooting and analyzing rows without matches

In [0]:
#Troubleshooting rows without matches
minor_ids = enrolled_as_minors['Subject ID'].unique()

# notes_enrolled Subject IDs not in REDCap — minors only
notes_no_match_minors = notes_enrolled[
    (~notes_enrolled['Subject ID'].isin(redcap_data['Record ID #:'])) & 
    (notes_enrolled['Subject ID'].isin(minor_ids))
]['Subject ID']
print(f"\nUnique minor Subject IDs in notes_enrolled with no match: {notes_no_match_minors.nunique()}")
print(sorted(notes_no_match_minors.unique()))

# REDCap Record IDs not in notes_enrolled — minors only
redcap_no_match_minors = redcap_data[
    (~redcap_data['Record ID #:'].isin(notes_enrolled['Subject ID'])) & 
    (redcap_data['Record ID #:'].isin(minor_ids))
]['Record ID #:']
print(f"Unique minor Record IDs in redcap_data with no match: {redcap_no_match_minors.nunique()}")
print(sorted(redcap_no_match_minors.unique()))

In [0]:
redcap_raw = pd.read_csv('/Workspace/Users/user/user/UndiagnosedMendelian-EHRExtraction_DATA_2026-05-05_1558.csv')

missing_ids = notes_no_match_minors.unique()
id_col = redcap_raw.columns[0]
found_ids = redcap_raw[redcap_raw[id_col].isin(missing_ids)][id_col].unique()

if len(found_ids) > 0:
    print(f"Found: {len(found_ids)}")
    print(sorted(found_ids))
else:
    print("None of the missing IDs were found in the raw file.")

In [0]:
#Checking author types for enrolled minors with no matching REDCap data
missing_minor_notes = notes_enrolled[notes_enrolled['Subject ID'].isin(notes_no_match_minors.unique())]
print(missing_minor_notes['AUTHOR_ROLE'].value_counts())

In [0]:
# Identifying duplicates for Subject ID in notes_enrolled2
id_counts = notes_enrolled2['Subject ID'].value_counts()
duplicates = id_counts[id_counts > 1]
print(f"Subject IDs with multiple notes: {len(duplicates)}")
print(duplicates)

In [0]:
# Read in scrambled patient data
REDCap_mapping_full_path = "/Workspace/Users/user/user/data_labels_filtered4.csv"

REDCap_mapping = pd.read_csv(REDCap_mapping_full_path)
print('Number of rows:', len(REDCap_mapping))
print('Number of columns:', len(REDCap_mapping.columns))
display(REDCap_mapping)

In [0]:
# Identifying duplicates for Record ID # in REDCap_mapping
REDCap_id_counts = REDCap_mapping['Record ID #:'].value_counts()
REDCap_duplicates = REDCap_id_counts[REDCap_id_counts > 1]
print(f"Record ID #s with multiple notes: {len(REDCap_duplicates)}")
REDCap_duplicates

In [0]:
# Identifying duplicates for Record ID # in redcap_data
redcap_data_id_counts = redcap_data['Record ID #:'].value_counts()
redcap_data_duplicates = redcap_data_id_counts[redcap_data_id_counts > 1]
print(f"Record ID #s with multiple notes: {len(redcap_data_duplicates)}")
redcap_data_duplicates

In [0]:
# For all duplicated Subject ID values in notes_enrolled2, how many unique values for column NOTE_FILE_TIME_DTTM does each unique duplicated Subject ID have?
dup_ids = notes_enrolled2['Subject ID'].value_counts()
dup_ids = dup_ids[dup_ids > 1].index

unique_dates = (
    notes_enrolled2[notes_enrolled2['Subject ID'].isin(dup_ids)]
    .groupby('Subject ID')['NOTE_FILE_TIME_DTTM']
    .nunique()
    .sort_values(ascending=False)
)
print(unique_dates)

## Dropping all notes except note right before most recent review from notes_enrolled2

In [0]:
# Ensure both columns are datetime for proper comparison
notes_enrolled3['NOTE_FILE_TIME_DTTM'] = pd.to_datetime(notes_enrolled3['NOTE_FILE_TIME_DTTM'])
notes_enrolled3['Date of last chart review:'] = pd.to_datetime(notes_enrolled3['Date of last chart review:'])

# Only keep rows where the note was filed on or before the chart review date
before_review = notes_enrolled3[
    notes_enrolled3['NOTE_FILE_TIME_DTTM'] <= notes_enrolled3['Date of last chart review:']
].copy()

# For each PAT_MRN_ID, keep the row with the most recent NOTE_FILE_TIME_DTTM
# (i.e., the closest date right before the chart review)
notes_enrolled4 = (
    before_review
    .sort_values('NOTE_FILE_TIME_DTTM', ascending=False)
    .drop_duplicates(subset='PAT_MRN_ID', keep='first')
    .reset_index(drop=True)
)

# --- Reporting ---
notes_enrolled3_mrns = set(notes_enrolled3['PAT_MRN_ID'].dropna().unique())
notes_enrolled4_mrns = set(notes_enrolled4['PAT_MRN_ID'].dropna().unique())
lost_mrns = notes_enrolled3_mrns - notes_enrolled4_mrns

print(f"notes_enrolled3 rows: {len(notes_enrolled3)}")
print(f"notes_enrolled3 unique MRNs: {len(notes_enrolled3_mrns)}")
print(f"Rows before review date: {len(before_review)}")
print(f"notes_enrolled4 rows: {len(notes_enrolled4)}")
print(f"notes_enrolled4 unique MRNs: {len(notes_enrolled4_mrns)}")
print(f"Unique MRNs lost: {len(lost_mrns)}")

In [0]:
# Checking to see if any MRNs lost to having their notes written after the date of last review
# have another entry with a later date of last review they could be matched on

# Get the notes for lost MRNs (all their notes were after their assigned review date)
notes_enrolled3_mrns = set(notes_enrolled3['PAT_MRN_ID'].dropna().unique())
notes_enrolled4_mrns = set(notes_enrolled4['PAT_MRN_ID'].dropna().unique())
lost_mrns = notes_enrolled3_mrns - notes_enrolled4_mrns

lost_notes = notes_enrolled3[
    notes_enrolled3['PAT_MRN_ID'].isin(lost_mrns)
]

# Get ALL review dates per MRN from notes_enrolled2 (before any dropping)
all_review_dates = notes_enrolled2[notes_enrolled2['Date of last chart review:'].notna()].copy()
all_review_dates['mrn_clean'] = strip_clean(all_review_dates['PAT_MRN_ID'])
all_review_dates['Date of last chart review:'] = pd.to_datetime(all_review_dates['Date of last chart review:'])
all_review_dates = all_review_dates[['mrn_clean', 'Date of last chart review:']].drop_duplicates()

# For each lost MRN, check if any later review date would allow a note to qualify
saveable = []
for mrn in lost_mrns:
    mrn_clean = strip_clean(pd.Series([mrn])).iloc[0]
    patient_notes = lost_notes[lost_notes['PAT_MRN_ID'] == mrn]['NOTE_FILE_TIME_DTTM']
    review_dates = all_review_dates[all_review_dates['mrn_clean'] == mrn_clean]['Date of last chart review:']
    
    for review_date in review_dates:
        qualifying_notes = patient_notes[patient_notes <= review_date]
        if len(qualifying_notes) > 0:
            saveable.append({
                'PAT_MRN_ID': mrn,
                'current_review_date': lost_notes[lost_notes['PAT_MRN_ID'] == mrn]['Date of last chart review:'].iloc[0],
                'alternative_review_date': review_date,
                'qualifying_notes_count': len(qualifying_notes),
                'closest_note': qualifying_notes.max()
            })

saveable_df = pd.DataFrame(saveable)
saveable_mrns = saveable_df['PAT_MRN_ID'].nunique() if len(saveable_df) > 0 else 0

print(f"Lost MRNs: {len(lost_mrns)}")
print(f"MRNs saveable with a later review date: {saveable_mrns}")
print(f"Still lost (no option): {len(lost_mrns) - saveable_mrns}")

if len(saveable_df) > 0:
    display(saveable_df)

In [0]:
print(f'Total unique MRNs started with:', notes_enrolled2['PAT_MRN_ID'].nunique())
print(f'Total unique MRNs after all filtering:', notes_enrolled4['PAT_MRN_ID'].nunique())
print(f'Total unique MRNs lost: {notes_enrolled2["PAT_MRN_ID"].nunique() - notes_enrolled4["PAT_MRN_ID"].nunique()}')

In [0]:
notes_enrolled4.to_csv("/Workspace/Users/user/user/notes_enrolled4.csv", index=False)
print("Saved notes_enrolled4.csv")

# Loading in and filtering checklist items

In [0]:
#Reading in the checklist of what features to search for
import pandas as pd

checklist_full_path = "/Workspace/Users/user/user/hposetta_stone2.csv"

checklist_full = pd.read_csv(checklist_full_path)
checklist_full.head(5)

In [0]:
import numpy as np
import pandas as pd

# clean up csv file by dropping all of the rows that we have decided to ignore
checklist_filtered = checklist_full[checklist_full['Ignore or not'] != 'ignore'].copy()
print(f"Before: {len(checklist_full)} rows")
print(f"After: {len(checklist_filtered)} rows")

# changing first column name to "Text from Checklist"
checklist_filtered.rename(columns={'HPO version: hpo-web@1.7.0 - hpo-obo@2020-08-11':'Text from Checklist'}, inplace=True)

# Get rid of the "Ignore or not" column
checklist_filtered.drop(columns=['Ignore or not'], inplace=True)
display(checklist_filtered)

In [0]:
i = 0 #row
j = 0 #column

# reformatting redcap file so that each row is a patient and successive columns contain phenotype
records = []
for i in range(0,len(REDCap_mapping)):
    phen = []
    # Rebecca had: for j in range(0,130): but this would exclude the last column?
    for j in range(0,len(REDCap_mapping.columns)):
        if pd.isnull(REDCap_mapping.iloc[i][j]) == False: 
            phen.append(REDCap_mapping.iloc[i][j])
    records.append(phen)

records_formatted = pd.DataFrame(records)
display(records_formatted)

In [0]:
manual_list = []
for j in range(0, len(records_formatted)): # rows
     for i in range(0, len(records_formatted.columns)): # columns
         manual_list.append(records_formatted[i][j])       

manual_list = [x for x in manual_list if x is not None]

display(manual_list)

In [0]:
hpocodes = []
for i in range(0, len(manual_list)):
    if manual_list[i][:3] == 'WES':
        id = manual_list[i]
    for j in range(0,len(checklist_filtered)): 
        # check if value is equal to "Term from checklist"
        if checklist_filtered.iloc[j][1] == (manual_list[i]): 
            #print(id) 
            #print(checklist_filtered.iloc[j][3])
            #print(id,checklist_filtered.iloc[j][3])
            manual_score = print(id, checklist_filtered.iloc[j][3])     

display(manual_score)

In [0]:
# Creates list of (Subject ID, HPO code) tuples
hpocodes_manual = []
# Loops through each item in manual_list; updates the current patient ID whenever a "WES" value appears
for i in range(0, len(manual_list)):
    if manual_list[i].startswith('WES'):
        id = manual_list[i]
    # For each item in manual_list, searches checklist_filtered for a matching term and appends the corresponding HPO code
    for j in range(0, len(checklist_filtered)):
        if checklist_filtered.iloc[j]['Term from checklist'] == manual_list[i]:
            hpocodes_manual.append((id, checklist_filtered.iloc[j]['HPO term (ID)']))

# Converts this list to a dataframe then displays it
manual_score = pd.DataFrame(hpocodes_manual, columns=['Subject ID', 'HPO term (ID)'])
display(manual_score)

In [0]:
rows = []
current_id = None
for item in manual_list:
    if item.startswith('WES'):
        current_id = item
    else:
        rows.append((current_id, item))

manual_df = pd.DataFrame(rows, columns=['Subject ID', 'Phenotype'])

manual_score = manual_df.merge(
    checklist_filtered[['Term from checklist', 'HPO term (ID)']],
    left_on='Phenotype',
    right_on='Term from checklist',
    how='inner'
).drop(columns=['Term from checklist'])
display(manual_score)

In [0]:
# Normalize phenotype names — strip whitespace once so downstream joins are clean
manual_score['Phenotype'] = manual_score['Phenotype'].str.strip()

In [0]:
manual_score.to_csv("/Workspace/Users/user/user/manual_score.csv", index=False)
print("Saved manual_score.csv")

# Preparing AI Extraction 

In [0]:
#Extracting unique phenotypes
print('Number of unique phenotypes:', checklist_filtered['Term from checklist'].nunique())
for term in checklist_filtered['Term from checklist'].unique():
    print(term)

In [0]:
display(checklist_filtered)

In [0]:
checklist_filtered.to_csv("/Workspace/Users/user/user/checklist_filtered.csv", index=False)
print("Saved checklist_filtered.csv")